# Week 2: Supervised Learning — Regression and Classification

## Mini Project 2: House Price Prediction
Download Kaggle House Prices `train.csv` and place it at `week2_supervised_learning/data/train.csv`. We create a holdout split because Kaggle's test file has no `SalePrice` labels. This notebook trains Linear Regression, Decision Tree, and Random Forest regressors, evaluates MSE and R², plots predicted versus actual prices, then demonstrates Titanic logistic regression.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if not (ROOT / 'week2_supervised_learning').exists(): ROOT = ROOT.parent
WEEK2 = ROOT / 'week2_supervised_learning'
HOUSING_PATH = WEEK2 / 'data' / 'train.csv'

## Load the housing data

In [ ]:
if not HOUSING_PATH.exists():
    raise FileNotFoundError(f'Download Kaggle train.csv and place it at {HOUSING_PATH}')
houses = pd.read_csv(HOUSING_PATH)
print('Shape:', houses.shape)
display(houses.head())
display(houses[['GrLivArea','OverallQual','GarageCars','TotalBsmtSF','YearBuilt','FullBath','SalePrice']].describe())

In [ ]:
features = ['GrLivArea','OverallQual','GarageCars','TotalBsmtSF','YearBuilt','FullBath']
X, y = houses[features], houses['SalePrice']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Training rows:', len(X_train), 'Test rows:', len(X_test))

## Train regression models and evaluate
MSE is the mean squared price error; R² summarizes how much target variation the model explains.

In [ ]:
models = {'Linear Regression': LinearRegression(), 'Decision Tree': DecisionTreeRegressor(max_depth=6, random_state=42), 'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)}
predictions = {}
for name, estimator in models.items():
    model = make_pipeline(SimpleImputer(strategy='median'), estimator)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    print(f'{name}: MSE={mean_squared_error(y_test, pred):,.2f}, R²={r2_score(y_test, pred):.4f}')

In [ ]:
actual = y_test.to_numpy()
predicted = predictions['Linear Regression']
low, high = min(actual.min(), predicted.min()), max(actual.max(), predicted.max())
plt.figure(figsize=(7,6))
plt.scatter(actual, predicted, alpha=0.65, color='teal', edgecolors='none')
plt.plot([low,high],[low,high],'r--',label='Perfect prediction')
plt.title('House Prices: Predicted vs Actual (Linear Regression)')
plt.xlabel('Actual sale price ($)'); plt.ylabel('Predicted sale price ($)')
plt.legend(); plt.tight_layout(); plt.show()

## Assignment: Logistic Regression on Titanic
Preprocessing is fit on training data only to avoid data leakage.

In [ ]:
titanic = pd.read_csv(ROOT / 'data' / 'Titanic-Dataset.csv')
num_cols = ['Pclass','Age','SibSp','Parch','Fare']; cat_cols = ['Sex','Embarked']
X_t = titanic[num_cols + cat_cols]; y_t = titanic['Survived']
Xtr, Xte, ytr, yte = train_test_split(X_t, y_t, test_size=0.2, random_state=42, stratify=y_t)
prep = ColumnTransformer([
 ('num', Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]), num_cols),
 ('cat', Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('encode',OneHotEncoder(handle_unknown='ignore'))]), cat_cols)])
clf = Pipeline([('prep',prep),('model',LogisticRegression(max_iter=1000,random_state=42))])
clf.fit(Xtr,ytr); pred = clf.predict(Xte)
print('Accuracy:', round(accuracy_score(yte,pred),4))
print('Confusion matrix (actual rows, predicted columns):\n', confusion_matrix(yte,pred,labels=[0,1]))